# Advanced feature extraction

This notebook is an alternative to `../2_feature_extraction.ipynb` for advanced usage. The standard notebook uses `RetinaLoader.from_folder(...)`; here we create `Retina` objects manually so custom per-image arguments can be passed into VascX.

The main example is `mm_per_pixel`. When provided, VascX uses it for physical ETDRS grid scaling and multiplies caliber and CRE biomarkers by that value. Leave it as `None` to keep the default pixel-based behavior.


In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
from rtnls_enface.bounds import make_roi_mask_from_bounds

from vascx.fundus.retina import Retina
from vascx.shared.features import FeatureSet
import vascx.fundus.feature_sets  # registers built-in fundus feature sets


## Input paths

The sample folder already contains preprocessed RGB images, artery-vein segmentations, vessel masks, disc masks, fovea coordinates, and preprocessing metadata. For your own data, point these paths at the output folder created by `vascx run-models`.


In [ ]:
ds_path = Path("../../samples/fundus")

rgb_dir = ds_path / "rgb"
av_dir = ds_path / "av"
vessels_dir = ds_path / "vessels"
disc_dir = ds_path / "discs"
fovea_csv = ds_path / "fovea.csv"
meta_csv = ds_path / "meta.csv"


## Optional mm_per_pixel values

Define `mm_per_pixel` values per image if your acquisition platform provides them. The values below are placeholders; replace them with real conversion factors for your dataset.


In [ ]:
# Example: provide real values here when available.
# manual_mm_per_pixel = {"DRIVE_22": 0.006, "DRIVE_40": 0.006}
manual_mm_per_pixel = {}
default_mm_per_pixel = None


## Build Retina objects manually

This is the key difference from the standard feature extraction notebook: each `Retina.from_file(...)` call can receive custom arguments. Here we pass `mm_per_pixel` when it is available.


In [ ]:
fovea_df = pd.read_csv(fovea_csv, index_col=0)
meta_df = pd.read_csv(meta_csv, index_col="id")

fovea_df.index = fovea_df.index.astype(str)
meta_df.index = meta_df.index.astype(str)

def parse_bounds(bounds_str):
    return eval(bounds_str, {"np": np})

def make_retina(image_id):
    mm_per_pixel = manual_mm_per_pixel.get(image_id, default_mm_per_pixel)
    bounds = parse_bounds(meta_df.loc[image_id, "bounds"])
    roi_mask = make_roi_mask_from_bounds(bounds, target_diameter=1024)

    return Retina.from_file(
        id=image_id,
        fundus_path=rgb_dir / f"{image_id}.png",
        av_path=av_dir / f"{image_id}.png",
        vessels_path=vessels_dir / f"{image_id}.png",
        disc_path=disc_dir / f"{image_id}.png",
        fovea_location=(
            float(fovea_df.loc[image_id, "mean_x"]),
            float(fovea_df.loc[image_id, "mean_y"]),
        ),
        roi_mask=roi_mask,
        mm_per_pixel=mm_per_pixel,
    )

image_ids = sorted(fovea_df.index)
retinas = [make_retina(image_id) for image_id in image_ids]
retinas[0]


## Compute biomarkers

Because we already created the `Retina` objects, feature extraction is just `retina.calc_features(...)`. This is easiest to customize, but for large datasets you may want to parallelize this pattern yourself.


In [ ]:
feature_set = FeatureSet.get_by_name("full_v3")

rows = []
for retina in retinas:
    row = retina.calc_features(feature_set)
    row["id"] = retina.id
    rows.append(row)

res = pd.DataFrame(rows).set_index("id")
res


## Save results

If `mm_per_pixel` was provided, caliber and CRE columns are in millimeters. Without it, they remain in pixels. Unitless biomarkers such as densities, ratios, angles, and tortuosity are not multiplied by `mm_per_pixel`.


In [ ]:
res.to_csv(ds_path / "biomarkers_advanced.csv")
